# OSCD learned pixel classifier (Phase D)

Train the optional per-pixel `HistGradientBoostingClassifier` on the 14 official **training** cities only. This notebook never loads, featurises or scores the 10 official test cities; `benchmarks/train_pixel_classifier.py` asserts that guard before reading any raster.

Run on a fresh Colab CPU runtime. Setup matches `notebooks/oscd_benchmark.ipynb`. Results are **unverified** until the owner pastes the output into the project; this notebook does not support any accuracy claim by itself.

Expected Drive layout (same data as the v2 benchmark):

```text
My Drive/oscd/
  archive (1).zip   # OSCD images and training labels
  test_labels.zip   # not used here; reserved for preregistered test scoring
```

The joblib artifact is written to Drive, outside git. Do not add a test split or a test-city score to this notebook.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/oscd'
REPO_URL = 'https://github.com/Aravind-Subramanian-16/satellite-change-detection.git'
REPO_DIR = '/content/repo'
OSCD_DIR = '/content/oscd'
SEED = 42

ARTIFACT_DIR = f'{DRIVE_DIR}/artifacts'
MODEL_PATH = f'{ARTIFACT_DIR}/learned_pixel_model.joblib'
IMAGES_ROOT = f'{OSCD_DIR}/images/Onera Satellite Change Detection dataset - Images'
TRAIN_LABELS_ROOT = f'{OSCD_DIR}/train_labels/Onera Satellite Change Detection dataset - Train Labels'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
archive = Path(DRIVE_DIR) / 'archive (1).zip'
if not archive.is_file():
    raise FileNotFoundError(f'Missing OSCD archive: {archive}')

!mkdir -p /content/oscd
!unzip -q -o "$archive" -d /content/oscd

if Path(REPO_DIR).is_dir():
    %cd /content/repo
    !git pull --ff-only
else:
    !git clone $REPO_URL $REPO_DIR
    %cd /content/repo

!python -m pip -q install -r requirements-benchmark.txt

!python -m pytest -q tests/test_learned_features.py tests/test_learned_pipeline.py

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/benchmarks')
from learned_features import OFFICIAL_TEST_CITIES, assert_no_test_cities
import oscd_eval

images = Path(IMAGES_ROOT)
train_cities = oscd_eval.read_city_list(images / 'train.txt')
if len(train_cities) != 14:
    raise ValueError(f'Expected 14 training cities, found {len(train_cities)}')
assert_no_test_cities(train_cities)
if set(train_cities) & set(OFFICIAL_TEST_CITIES):
    raise ValueError('Training split contains an official test city')

labels_root = Path(TRAIN_LABELS_ROOT)
for city in train_cities:
    for directory in (images / city / 'imgs_1_rect', images / city / 'imgs_2_rect'):
        if not directory.is_dir():
            raise FileNotFoundError(f'{city}: missing acquisition at {directory}')
    label = labels_root / city / 'cm' / f'{city}-cm.tif'
    if not label.is_file():
        raise FileNotFoundError(f'{city}: missing training label at {label}')

print(f'Verified {len(train_cities)} TRAIN cities: {", ".join(train_cities)}')
print('Official test cities are excluded by the guard and are not read.')

In [ ]:
%cd /content/repo
!python benchmarks/train_pixel_classifier.py \
  --images-root "$IMAGES_ROOT" \
  --train-labels-root "$TRAIN_LABELS_ROOT" \
  --output-model "$MODEL_PATH" \
  --seed $SEED

The command above prints the three seeded held-out training cities, the selected probability threshold and the held-out validation metrics. Those numbers describe **training-city validation only**. No test-city score exists in this notebook; do not describe them as test accuracy, and do not tune after seeing test cities.

In [ ]:
import json
import joblib

bundle = joblib.load(MODEL_PATH)
metadata = bundle['metadata']
if metadata.get('test_split_scored') is not False:
    raise ValueError('Artifact metadata must state test_split_scored: false')
if metadata.get('official_test_cities_excluded') != list(OFFICIAL_TEST_CITIES):
    raise ValueError('Artifact does not record the official test-city exclusion list')
if len(metadata.get('fit_cities', [])) != 11 or len(metadata.get('held_out_train_cities', [])) != 3:
    raise ValueError('Expected 11 fit cities and 3 held-out TRAIN cities')

summary = {key: metadata[key] for key in (
    'created_utc', 'code_commit', 'seed', 'fit_cities', 'held_out_train_cities',
    'fit_pixels', 'held_out_pixels', 'fit_change_percent', 'held_out_change_percent',
    'feature_count', 'hyperparameters', 'selected_threshold',
    'validation_metrics_at_selected_threshold', 'validation_metrics_at_0_5',
    'test_split_scored')}
print(json.dumps(summary, indent=2))
print(f'Artifact: {MODEL_PATH} ({Path(MODEL_PATH).stat().st_size:,} bytes)')
print('Paste the held-out TRAIN validation summary above into the project; it is not a test result.')

## Next steps

- Paste the training output and the `test_split_scored: false` artifact summary into the project so the validation numbers are recorded as owner-verified Colab evidence.
- Keep the joblib on Drive and outside git. To serve it, copy it to the deployment host and set `LEARNED_MODEL_PATH` plus `ENABLE_LEARNED_MODEL=1`; the spectral/live pipeline remains the default.
- Test-city scoring is deferred to Phase D2.6 and may run only once for systems listed in `docs/paper/PREREGISTRATION.md`. This notebook must not be edited to score the test split.